# Chapter 5: Proximal Policy Optimization
### **RL: The Seminal Papers** by Rahul Shirale

Welcome to the interactive companion notebook for Chapter 5. We implement **PPO** from Schulman et al. (2017), "Proximal Policy Optimization Algorithms." PPO achieves the stability of Trust Region Policy Optimization (TRPO) using only first-order optimization, via a deceptively simple **clipped surrogate objective** that prevents the policy from changing too much in a single update. We train a full PPO agent with Generalized Advantage Estimation (GAE) on `Pendulum-v1`, reaching a competent policy in under ten minutes on a standard CPU.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rshirale/rl-seminal-papers/blob/main/src/part_2_methods/ch05_ppo/Chapter5_PPO.ipynb)

## 1. Setup
The cell below installs dependencies. In Google Colab, uncomment and run it. Locally, use `make install-full` from the repo root.

In [ ]:
# Uncomment in Google Colab. The specifiers are quoted because an
# unquoted ">=" is read by the shell as a redirection.
# %pip install "torch>=2.0.0" "gymnasium[classic-control]>=1.0,<2.0" matplotlib numpy

import os

import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym
import torch
import torch.nn as nn
from torch.distributions import Normal

print(f"PyTorch: {torch.__version__} | Gymnasium: {gym.__version__}")

## 2. Why Vanilla Policy Gradients Fail

Policy gradient methods like REINFORCE directly optimise the policy $\pi_\theta$ by following the gradient of expected return:

$$\nabla_\theta J(\theta) = \mathbb{E}_t \left[ \nabla_\theta \log \pi_\theta(a_t|s_t) \cdot \hat{A}_t \right]$$

The fatal flaw is **step size sensitivity**. In supervised learning, a bad gradient step temporarily spikes the loss, but the next mini-batch of i.i.d. data corrects it. In RL, the agent **generates its own training data**. A policy that is slightly too bad explores poorly, collects low-quality data, and makes the next update even worse. The agent can fall into a performance collapse from which it cannot escape without a full restart.

The plot below illustrates the relationship between step size and training stability.

In [ ]:
np.random.seed(0)
episodes = np.arange(1, 201)

# Simulated learning curves for three step-size regimes
def smooth_curve(start, end, n, noise_scale, collapse_at=None):
    x = np.linspace(start, end, n) + np.random.randn(n) * noise_scale
    if collapse_at is not None:
        x[collapse_at:] = start + np.random.randn(n - collapse_at) * noise_scale * 2
    return x

good_lr   = smooth_curve(-1400, -250, 200, 80)
small_lr  = smooth_curve(-1400, -900, 200, 60)
large_lr  = smooth_curve(-1400, -600, 200, 100, collapse_at=60)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(episodes, large_lr,  color="#CC3311", alpha=0.8, label="Too large — policy collapse at ~episode 60")
ax.plot(episodes, small_lr,  color="#EE7733", alpha=0.8, linestyle="--", label="Too small — very slow progress")
ax.plot(episodes, good_lr,   color="#009988", linewidth=2, label="PPO clipping — stable improvement")
ax.axhline(-200, linestyle=":", color="#555555", linewidth=1, label="Competent threshold (\u2212200)")
ax.set_xlabel("Episode")
ax.set_ylabel("Episodic Return")
ax.set_title("The step-size dilemma: too large collapses the policy; too small learns nothing")
ax.legend(loc="lower right", fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. From TRPO to PPO

Trust Region Policy Optimization (TRPO, Schulman et al. 2015) solved the instability problem with a rigorous mathematical constraint: the new policy $\pi_\theta$ must stay within a **trust region** defined by a maximum KL divergence from the old policy $\pi_{\theta_{old}}$:

$$\text{maximise } \hat{\mathbb{E}}_t \left[ \frac{\pi_\theta(a_t|s_t)}{\pi_{\theta_{old}}(a_t|s_t)} \hat{A}_t \right] \quad \text{subject to } \hat{\mathbb{E}}_t \left[ \text{KL}[\pi_{\theta_{old}}, \pi_\theta] \right] \leq \delta$$

TRPO works well but requires second-order optimization (conjugate gradients, Fisher Information Matrix), making it slow and hard to implement — especially with shared actor-critic architectures.

**PPO's insight:** the trust region constraint can be approximated by simply *clipping* the probability ratio. No second-order methods needed.

## 4. The Clipped Surrogate Objective

The central innovation of PPO is replacing the constrained TRPO objective with a clipped one. Define the **probability ratio**:

$$r_t(\theta) = \frac{\pi_\theta(a_t|s_t)}{\pi_{\theta_{old}}(a_t|s_t)}$$

When $r_t(\theta) = 1$, the policy is unchanged. The PPO clipped objective is:

$$L^{\text{CLIP}}(\theta) = \hat{\mathbb{E}}_t \left[ \min\left(r_t(\theta)\hat{A}_t,\ \text{clip}(r_t(\theta), 1-\varepsilon, 1+\varepsilon)\hat{A}_t\right) \right]$$

The `min` ensures we take the **pessimistic bound**: if the advantage is positive, we stop rewarding the network for increasing the ratio beyond $1+\varepsilon$; if negative, we stop penalising beyond $1-\varepsilon$. The clipped objective is flat outside the trust region — there is literally no gradient to push the policy further away.

The cell below plots this behaviour directly.

In [ ]:
eps = 0.2
r   = np.linspace(0.5, 1.5, 300)

def l_clip(r, A, eps):
    unclipped = r * A
    clipped   = np.clip(r, 1 - eps, 1 + eps) * A
    return np.minimum(unclipped, clipped)

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=False)

for ax, A, title in zip(
    axes,
    [1.0, -1.0],
    ["Positive advantage ($\\hat{A}_t > 0$): good action, increase probability",
     "Negative advantage ($\\hat{A}_t < 0$): bad action, decrease probability"]
):
    ax.plot(r, r * A,               color="#BBBBBB", linewidth=1.5, linestyle="--", label="Unclipped $r_t \\hat{A}_t$")
    ax.plot(r, l_clip(r, A, eps),   color="#5A4FCF", linewidth=2.5, label="$L^{\\mathrm{CLIP}}$ (PPO objective)")
    ax.axvline(1 - eps, color="#F0A500", linestyle=":", linewidth=1.2, label=f"$1-\\varepsilon = {1-eps}$")
    ax.axvline(1 + eps, color="#F0A500", linestyle=":", linewidth=1.2, label=f"$1+\\varepsilon = {1+eps}$")
    ax.axvline(1.0,     color="#888888", linestyle="-",  linewidth=0.8)
    ax.set_xlabel("Probability ratio $r_t(\\theta)$")
    ax.set_ylabel("Objective value")
    ax.set_title(title, fontsize=10)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle("PPO Clipped Surrogate Objective ($\\varepsilon = 0.2$)", fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

## 5. Generalized Advantage Estimation (GAE)

The clipped objective stabilises the *policy update*, but we also need a stable *advantage estimate* $\hat{A}_t$. The advantage tells us how much better or worse action $a_t$ was compared to the average: $A(s,a) = Q(s,a) - V(s)$.

Two extremes exist:
- **Monte Carlo** (full episode return): unbiased but very high variance.
- **1-step TD error** ($\delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)$): low variance but biased.

GAE (Schulman et al. 2016) interpolates between them with a hyperparameter $\lambda \in [0,1]$:

$$\hat{A}_t^{\text{GAE}(\gamma,\lambda)} = \sum_{l=0}^{\infty} (\gamma\lambda)^l \delta_{t+l}$$

In practice this is computed backwards through the rollout:
$$\text{gae}_t = \delta_t + \gamma\lambda(1-d_t)\cdot\text{gae}_{t+1}$$

- $\lambda = 0$: pure TD error (low variance, high bias)
- $\lambda = 1$: Monte Carlo (zero bias, high variance)
- $\lambda = 0.95$: the PPO default — a well-tuned tradeoff that remains the standard in RLHF pipelines today.

## 6. Actor and Critic Networks

Unlike DDPG's deterministic actor, PPO requires a **stochastic policy** to enable exploration and to compute the log-probabilities needed for the ratio $r_t(\theta)$. The `Actor` outputs the parameters of a Gaussian distribution: mean $\mu$ and standard deviation $\sigma$. The agent samples its action from $\mathcal{N}(\mu, \sigma^2)$.

A design choice worth noting: `log_std` is a single learned `nn.Parameter` independent of the state. This simplifies training while maintaining effective exploration — and it is the dominant choice in production PPO implementations.

In [ ]:
class Actor(nn.Module):
    def __init__(self, state_dim, action_dim, max_action):
        super(Actor, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, 64), nn.Tanh(),
            nn.Linear(64, 64),        nn.Tanh()
        )
        self.mu = nn.Linear(64, action_dim)
        self.log_std = nn.Parameter(torch.zeros(1, action_dim))
        self.max_action = max_action

    def forward(self, state):
        x = self.net(state)
        mu = torch.tanh(self.mu(x)) * self.max_action
        std = self.log_std.exp().expand_as(mu)
        return Normal(mu, std)


class Critic(nn.Module):
    def __init__(self, state_dim):
        super(Critic, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, 64), nn.Tanh(),
            nn.Linear(64, 64),        nn.Tanh(),
            nn.Linear(64, 1)
        )

    def forward(self, state):
        return self.net(state)


# Smoke test — Pendulum-v1: state_dim=3, action_dim=1, max_action=2.0
actor_test = Actor(state_dim=3, action_dim=1, max_action=2.0)
critic_test = Critic(state_dim=3)
s = torch.zeros(1, 3)
dist = actor_test(s)
print(f"Actor: distribution mean={dist.mean.item():.4f}, std={dist.stddev.item():.4f}")
print(f"Critic: V(s)={critic_test(s).item():.4f}")

## 7. The PPO Agent

The `PPOAgent` class assembles three components:
- **`Actor`** and **`Critic`** networks sharing a single Adam optimizer.
- **`select_action`**: samples a stochastic action and records its log-probability and value estimate for the update step.
- **`update`**: iterates backwards through the rollout to compute GAE, normalises advantages, then runs $K$ epochs of minibatch SGD on the clipped objective.

Two production details are included that the book's teaching snippets omit:
- **Minibatch SGD** (batch size 64): reduces gradient variance within each epoch.
- **Gradient clipping** (`max_norm=0.5`): prevents occasional large gradient steps from destabilising training.

In [ ]:
class PPOAgent:
    def __init__(self, state_dim, action_dim, max_action,
                 lr=1e-3, gamma=0.9, lam=0.95,
                 eps_clip=0.2, k_epochs=10, batch_size=64):
        self.gamma = gamma
        self.lam = lam
        self.eps_clip = eps_clip
        self.k_epochs = k_epochs
        self.batch_size = batch_size

        self.actor = Actor(state_dim, action_dim, max_action)
        self.critic = Critic(state_dim)
        self.optimizer = torch.optim.Adam(
            list(self.actor.parameters()) + list(self.critic.parameters()),
            lr=lr
        )

    def select_action(self, state):
        state_t = torch.FloatTensor(state).unsqueeze(0)
        with torch.no_grad():
            dist = self.actor(state_t)
            value = self.critic(state_t).item()
        action = dist.sample()
        logprob = dist.log_prob(action).sum(dim=-1).item()
        return action.squeeze(0).numpy(), logprob, value

    def update(self, rollouts):
        states, actions, rewards, next_states, dones, old_logprobs, values = zip(*rollouts)

        states = torch.FloatTensor(np.array(states))
        actions = torch.FloatTensor(np.array(actions))
        old_logprobs = torch.FloatTensor(np.array(old_logprobs))
        rewards, dones, values = list(rewards), list(dones), list(values)

        # --- GAE backwards pass ---
        returns, advantages, gae = [], [], 0.0
        with torch.no_grad():
            next_value = self.critic(
                torch.FloatTensor(next_states[-1]).unsqueeze(0)
            ).item()

        for i in reversed(range(len(rollouts))):
            next_val = next_value if i == len(rollouts) - 1 else values[i + 1]
            delta = rewards[i] + self.gamma * next_val * (1 - dones[i]) - values[i]
            gae = delta + self.gamma * self.lam * (1 - dones[i]) * gae
            advantages.insert(0, gae)
            returns.insert(0, gae + values[i])

        advantages = torch.FloatTensor(np.array(advantages))
        returns = torch.FloatTensor(np.array(returns))
        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

        # --- K epochs of minibatch SGD ---
        n = len(rollouts)
        approx_kl_sum, clip_frac_sum, count = 0.0, 0.0, 0

        for _ in range(self.k_epochs):
            perm = np.random.permutation(n)
            for start in range(0, n, self.batch_size):
                idx = perm[start: start + self.batch_size]
                mb_s, mb_a = states[idx], actions[idx]
                mb_old_lp = old_logprobs[idx]
                mb_adv = advantages[idx]
                mb_ret = returns[idx]

                dist = self.actor(mb_s)
                logprobs = dist.log_prob(mb_a).sum(dim=-1)
                entropy = dist.entropy().sum(dim=-1)
                state_values = self.critic(mb_s).squeeze(-1)

                ratios = torch.exp(logprobs - mb_old_lp)
                surr1 = ratios * mb_adv
                surr2 = torch.clamp(ratios, 1 - self.eps_clip, 1 + self.eps_clip) * mb_adv

                loss = (
                    -torch.min(surr1, surr2)
                    + 0.5 * nn.MSELoss()(state_values, mb_ret)
                    - 0.01 * entropy
                )

                self.optimizer.zero_grad()
                loss.mean().backward()
                nn.utils.clip_grad_norm_(
                    list(self.actor.parameters()) + list(self.critic.parameters()),
                    max_norm=0.5
                )
                self.optimizer.step()

                with torch.no_grad():
                    approx_kl_sum += (mb_old_lp - logprobs).mean().item()
                    clip_frac_sum += ((ratios - 1.0).abs() > self.eps_clip).float().mean().item()
                    count += 1

        return approx_kl_sum / count, clip_frac_sum / count


print("PPOAgent defined.")

## 8. Training on Pendulum-v1

The environment `Pendulum-v1` is a canonical continuous-control benchmark. The 3-element state vector is $(\cos\theta, \sin\theta, \dot\theta)$ and the single continuous action is torque $\in [-2, 2]$. The reward penalises angular distance from vertical, angular velocity, and torque: $r \approx -(\theta^2 + 0.1\dot\theta^2 + 0.001\tau^2)$. A competent agent holds the pendulum upright, achieving episode returns above $-200$.

Unlike DQN and DDPG, PPO is **on-policy**: the rollout buffer is discarded after each update. Data collected from a previous version of the policy cannot be reused.

In [ ]:
env_preview = gym.make("Pendulum-v1", render_mode="rgb_array")
env_preview.reset(seed=42)
frame = env_preview.render()
env_preview.close()

env_info = gym.make("Pendulum-v1")
print("Observation space:", env_info.observation_space)
print("Action space:     ", env_info.action_space)
print(f"Max action:        {float(env_info.action_space.high[0]):.1f}")
env_info.close()

fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(frame)
ax.axis("off")
ax.set_title("Pendulum-v1 — swing up and balance using continuous torque")
plt.tight_layout()
plt.show()

In [ ]:
from IPython.display import clear_output

SEED = 42
# Overridable so the test suite can execute this notebook quickly.
MAX_EPISODES = int(os.environ.get("CH5_NUM_EPISODES", 400))
MAX_TIMESTEPS = 200
UPDATE_EVERY = 2048
LOG_EVERY = 20

torch.manual_seed(SEED)
np.random.seed(SEED)

env = gym.make("Pendulum-v1")
state_dim = env.observation_space.shape[0]
action_dim = env.action_space.shape[0]
max_action = float(env.action_space.high[0])

agent = PPOAgent(
    state_dim, action_dim, max_action,
    lr=1e-3, gamma=0.9, lam=0.95,
    eps_clip=0.2, k_epochs=10, batch_size=64
)

rollouts = []
timestep = 0
ep_rewards = []
approx_kl = 0.0
clip_frac = 0.0
kl_history = []
cf_history = []

for episode in range(1, MAX_EPISODES + 1):
    state, _ = env.reset(seed=SEED + episode)
    ep_reward = 0.0

    for t in range(MAX_TIMESTEPS):
        timestep += 1
        action, logprob, value = agent.select_action(state)
        next_state, reward, done, truncated, _ = env.step(action)

        # True termination stops bootstrapping; time-limit truncation does not.
        rollouts.append((state, action, reward, next_state,
                         float(done), logprob, value))
        state = next_state
        ep_reward += reward

        if timestep % UPDATE_EVERY == 0:
            approx_kl, clip_frac = agent.update(rollouts)
            rollouts = []
            kl_history.append(approx_kl)
            cf_history.append(clip_frac)

        if done or truncated:
            break

    ep_rewards.append(ep_reward)

    if episode % LOG_EVERY == 0:
        avg = np.mean(ep_rewards[-LOG_EVERY:])
        clear_output(wait=True)
        window = 20
        rolling = [
            np.mean(ep_rewards[max(0, i - window + 1): i + 1])
            for i in range(len(ep_rewards))
        ]
        fig, ax = plt.subplots(figsize=(10, 3))
        ax.plot(ep_rewards, alpha=0.2, color="#5A4FCF", label="Episode return")
        ax.plot(rolling, linewidth=2, color="#F0A500", label=f"{window}-ep moving avg")
        ax.axhline(-200, linestyle="--", color="#009988", linewidth=1,
                   label="Competent threshold (\u2212200)")
        ax.set_title(
            f"PPO Training \u2014 Episode {episode}/{MAX_EPISODES}"
            f"  |  Last-{LOG_EVERY} avg: {avg:.0f}"
            f"  |  approx_kl: {approx_kl:.3f}  clip_frac: {clip_frac:.2f}"
        )
        ax.set_xlabel("Episode")
        ax.set_ylabel("Return")
        ax.legend(loc="upper left")
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

env.close()
print("Training complete.")

## 9. Diagnostics: Trust-Region Metrics

PPO exposes two metrics that let you verify the trust region is working:

- **`approx_kl`**: approximate KL divergence between old and new policy. Healthy range: 0.005–0.03. Consistently above 0.05 means the update is too large.
- **`clip_frac`**: fraction of transitions where the ratio was clipped. Healthy range: 0.08–0.20. Above 0.3 means clipping is over-constraining learning.

The plots below confirm these diagnostics over the training run.

In [ ]:
window = 20
rolling = [
    np.mean(ep_rewards[max(0, i - window + 1): i + 1])
    for i in range(len(ep_rewards))
]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Learning curve
axes[0].plot(ep_rewards, alpha=0.2, color="#5A4FCF", label="Episode return")
axes[0].plot(rolling, linewidth=2, color="#F0A500", label=f"{window}-ep moving avg")
axes[0].axhline(-200, linestyle="--", color="#009988", linewidth=1, label="Competent (\u2212200)")
axes[0].set_xlabel("Episode")
axes[0].set_ylabel("Return")
axes[0].set_title("PPO on Pendulum-v1")
axes[0].legend(loc="lower right", fontsize=8)
axes[0].grid(True, alpha=0.3)

# approx_kl
update_steps = np.arange(1, len(kl_history) + 1)
axes[1].plot(update_steps, kl_history, color="#CC3311", linewidth=1.5)
axes[1].axhline(0.05, linestyle="--", color="#888888", linewidth=1, label="Warning threshold (0.05)")
axes[1].set_xlabel("Update step")
axes[1].set_ylabel("approx_kl")
axes[1].set_title("KL divergence per update")
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

# clip_frac
axes[2].plot(update_steps, cf_history, color="#117733", linewidth=1.5)
axes[2].axhline(0.3, linestyle="--", color="#888888", linewidth=1, label="Warning threshold (0.30)")
axes[2].set_xlabel("Update step")
axes[2].set_ylabel("clip_frac")
axes[2].set_title("Clipping fraction per update")
axes[2].legend(fontsize=8)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

final_avg = np.mean(ep_rewards[-20:])
print(f"Final 20-episode average: {final_avg:.1f}")
print(f"Competent threshold: \u2212200  |  {'PASSED ✓' if final_avg > -200 else 'not yet reached'}")

## 10. Using the Module Files

The notebook is self-contained for learning. In a production setting, import from the companion scripts:

In [ ]:
# If running locally from the ch05_ppo directory:
# from actor_critic import Actor, Critic
# from ppo_agent import PPOAgent
#
# To run the full Pendulum-v1 training from the terminal:
#   python src/part_2_methods/ch05_ppo/train_pendulum.py
#
# To import from the package (repo root must be on PYTHONPATH):
#   from src.part_2_methods.ch05_ppo import PPOAgent